# Playlist Matcher Test Notebook

This notebook creates a mock music library with the first 10 songs from Favourites.m3u8 and tests the playlist_matcher.py script.

## Setup

First, we'll install the required dependencies and import necessary modules.

In [1]:
# Install mutagen if not already installed
!pip install mutagen


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import shutil
from pathlib import Path
from mutagen.flac import FLAC
import re
import struct
from importlib import reload

import playlist_matcher as pm

## Helper Functions

Functions to sanitize filenames and handle special characters.

In [3]:
def sanitize_filename(name):
    """Sanitize a string for use in filenames by replacing problematic characters"""
    # Replace forward slash with unicode division slash or underscore
    # This preserves the visual appearance while being filesystem-safe
    name = name.replace('/', '∕')  # Unicode division slash U+2215
    # Could also use: name = name.replace('/', '_')
    
    # Replace other problematic characters
    replacements = {
        '\\': '∖',  # Backslash
        ':': '∶',   # Colon  
        '*': '∗',   # Asterisk
        '?': '？',  # Question mark
        '"': '＂',  # Quote
        '<': '＜',  # Less than
        '>': '＞',  # Greater than
        '|': '｜',  # Pipe
    }
    
    for old, new in replacements.items():
        name = name.replace(old, new)
    
    return name

def sanitize_path_component(name):
    """Sanitize directory/album names"""
    return sanitize_filename(name)

## Parse First 10 Entries from Favourites.m3u8

We'll read the playlist and extract metadata for the first 10 songs.

In [4]:
def parse_playlist_entries(playlist_path, num_entries=10):
    """Parse first N entries from playlist"""
    entries = []
    
    with open(playlist_path, 'r', encoding='utf-8-sig') as f:
        lines = [line.rstrip('\n\r') for line in f.readlines()]
    
    i = 0
    count = 0
    while i < len(lines) and count < num_entries:
        line = lines[i]
        
        if line.startswith('#EXTINF:'):
            if i + 1 < len(lines):
                extinf_line = line
                path_line = lines[i + 1]
                
                # Parse EXTINF: #EXTINF:duration,Artist - Title
                extinf_match = re.match(r'#EXTINF:(\d+),(.+)', extinf_line)
                if extinf_match:
                    duration = extinf_match.group(1)
                    artist_title = extinf_match.group(2)
                    
                    # Split artist and title
                    if ' - ' in artist_title:
                        artist, title = artist_title.split(' - ', 1)
                    else:
                        artist = ""
                        title = artist_title
                    
                    # Parse path: ..\Artist(s)\Album\CD# - Track# - Artist(s) - Title - Album.ext
                    clean_path = path_line.replace('..\\', '').replace('../', '').replace('\\', '/')
                    path_parts = clean_path.split('/')
                    
                    playlist_artist = path_parts[0] if len(path_parts) > 0 else artist
                    album = path_parts[1] if len(path_parts) > 1 else "Unknown Album"
                    filename = path_parts[2] if len(path_parts) > 2 else "unknown.flac"
                    
                    # Parse filename for additional metadata
                    # Format: CD# - Track# - Artist(s) - Title - Album.ext
                    # Use ' - ' (space-dash-space) as delimiter to allow '-' in names
                    filename_match = re.match(r'^(\d+) - (\d+) - (.+?) - (.+?) - (.+?)\.(\w+)$', filename)
                    
                    if filename_match:
                        disc = filename_match.group(1)
                        track = filename_match.group(2)
                        file_artist = filename_match.group(3)
                        file_title = filename_match.group(4)
                        file_album = filename_match.group(5)
                        ext = filename_match.group(6)
                    else:
                        disc = "1"
                        track = str(count + 1)
                        file_artist = artist
                        file_title = title
                        file_album = album
                        ext = "flac"
                    
                    entries.append({
                        'artist': artist.strip(),
                        'title': title.strip(),
                        'album': album.strip(),
                        'playlist_artist': playlist_artist.strip(),
                        'file_artist': file_artist.strip(),
                        'file_title': file_title.strip(),
                        'file_album': file_album.strip(),
                        'disc': disc,
                        'track': track,
                        'ext': ext,
                        'duration': duration,
                        'original_path': path_line
                    })
                    count += 1
                
                i += 2
            else:
                i += 1
        else:
            i += 1
    
    return entries

# Parse first 10 entries
entries = parse_playlist_entries('../Playlists/Favourites.m3u8', 10)

print(f"Parsed {len(entries)} entries:\n")
for i, entry in enumerate(entries, 1):
    print(f"{i}. {entry['artist']} - {entry['title']}")
    print(f"   Album: {entry['album']}")
    print(f"   Track: {entry['disc']}-{entry['track']}")
    print(f"   File title: {entry['file_title']}")
    print()

Parsed 10 entries:

1. The Offspring - (Can't Get My) Head Around You
   Album: Splinter
   Track: 1-164
   File title: (Can't Get My) Head Around You

2. Santana - (Da Le) Yaleo
   Album: Supernatural (Remastered)
   Track: 1-208
   File title: (Da Le) Yaleo

3. Jamiroquai - (Don't) Give Hate a Chance
   Album: Dynamite
   Track: 1-247
   File title: (Don't) Give Hate a Chance

4. The Rolling Stones - (I Can't Get No) Satisfaction - Mono
   Album: Out Of Our Heads
   Track: 1-458
   File title: (I Can't Get No) Satisfaction

5. JAY-Z, Beyoncé - 03' Bonnie & Clyde
   Album: The Blueprint 2 The Gift & The Curse
   Track: 1-1
   File title: 03' Bonnie & Clyde

6. Die Ärzte - 1/2 Lovesong
   Album: 13
   Track: 1-2
   File title: 12 Lovesong

7. Ciara, Missy Elliott - 1, 2 Step (feat. Missy Elliott)
   Album: Goodies
   Track: 1-3
   File title: 1, 2 Step (feat. Missy Elliott)

8. Gorillaz - 19-2000 - Soulchild Remix
   Album: Gorillaz
   Track: 1-4
   File title: 19-2000

9. A Tribe Call

## Create Mock Library Using Template

Now we'll copy the template FLAC file and modify its metadata for each song.
We sanitize filenames to handle special characters like '/' safely.

In [5]:
def create_mock_library(entries, template_path='example.flac', base_dir='test_music_library'):
    """Create mock music library by copying template and modifying metadata"""
    
    if not os.path.exists(template_path):
        print(f"Error: Template file {template_path} not found!")
        return []
    
    # Clean up existing directory
    if os.path.exists(base_dir):
        shutil.rmtree(base_dir)
    
    os.makedirs(base_dir)
    
    created_files = []
    
    for entry in entries:
        # Use artist as album artist for the directory structure
        album_artist = entry['artist']
        album = entry['album']
        
        # Sanitize directory names
        safe_artist = sanitize_path_component(album_artist)
        safe_album = sanitize_path_component(album)
        
        # Create directory structure
        album_dir = Path(base_dir) / safe_artist / safe_album
        album_dir.mkdir(parents=True, exist_ok=True)
        
        # Create filename in library format with sanitized components
        # Format: CD# - Track# - Title - Artist(s) - Album.ext
        safe_title = sanitize_filename(entry['title'])
        safe_artist_name = sanitize_filename(entry['artist'])
        safe_album_name = sanitize_filename(entry['album'])
        
        filename = f"{entry['disc']} - {entry['track']} - {safe_title} - {safe_artist_name} - {safe_album_name}.{entry['ext']}"
        file_path = album_dir / filename
        
        try:
            # Copy template file
            shutil.copy2(template_path, file_path)
            
            # Modify metadata - use ORIGINAL unsanitized values for metadata tags
            audio = FLAC(str(file_path))
            audio['title'] = entry['title']  # Original with /
            audio['artist'] = entry['artist']
            audio['album'] = entry['album']
            audio['albumartist'] = album_artist
            audio['tracknumber'] = entry['track']
            audio['discnumber'] = entry['disc']
            audio.save()
            
            # Verify
            audio = FLAC(str(file_path))
            print(f"✓ Created: {file_path.relative_to(base_dir)}")
            print(f"  Metadata: '{audio.get('title', [''])[0]}' by '{audio.get('artist', [''])[0]}'")
            
            created_files.append(str(file_path))
            
        except Exception as e:
            print(f"✗ Failed to create: {file_path.relative_to(base_dir) if file_path.exists() else filename}")
            print(f"  Error: {e}")
            import traceback
            traceback.print_exc()
    
    return created_files

# Create the mock library
print("Creating mock music library from template...\n")
created_files = create_mock_library(entries)
print(f"\nCreated {len(created_files)} files in test_music_library/")

Creating mock music library from template...

✓ Created: The Offspring\Splinter\1 - 164 - (Can't Get My) Head Around You - The Offspring - Splinter.flac
  Metadata: '(Can't Get My) Head Around You' by 'The Offspring'
✓ Created: Santana\Supernatural (Remastered)\1 - 208 - (Da Le) Yaleo - Santana - Supernatural (Remastered).flac
  Metadata: '(Da Le) Yaleo' by 'Santana'
✓ Created: Jamiroquai\Dynamite\1 - 247 - (Don't) Give Hate a Chance - Jamiroquai - Dynamite.flac
  Metadata: '(Don't) Give Hate a Chance' by 'Jamiroquai'
✓ Created: The Rolling Stones\Out Of Our Heads\1 - 458 - (I Can't Get No) Satisfaction - Mono - The Rolling Stones - Out Of Our Heads.flac
  Metadata: '(I Can't Get No) Satisfaction - Mono' by 'The Rolling Stones'
✓ Created: JAY-Z, Beyoncé\The Blueprint 2 The Gift & The Curse\1 - 1 - 03' Bonnie & Clyde - JAY-Z, Beyoncé - The Blueprint 2 The Gift & The Curse.flac
  Metadata: '03' Bonnie & Clyde' by 'JAY-Z, Beyoncé'
✓ Created: Die Ärzte\13\1 - 2 - 1∕2 Lovesong - Die Ärzte -

## Create Test Playlist

Create a small test playlist with just these 10 songs.

In [6]:
def create_test_playlist(entries, output_path='test_playlist.m3u8'):
    """Create a test playlist with the parsed entries"""
    with open(output_path, 'w', encoding='utf-8') as f:
        f.write('#EXTM3U\n')
        for entry in entries:
            f.write(f"#EXTINF:{entry['duration']},{entry['artist']} - {entry['title']}\n")
            f.write(f"{entry['original_path']}\n")
    
    print(f"Created test playlist: {output_path}")

create_test_playlist(entries)

Created test playlist: test_playlist.m3u8


## Test the Playlist Matcher Script

Now let's run the playlist matcher script on our test data.

In [7]:
# Run the playlist matcher
!python playlist_matcher.py \
    --playlist test_playlist.m3u8 \
    --music-dir test_music_library \
    --output test_output.m3u8 \
    --log test_unmatched.log \
    --format artist_album

2026-05-08 09:58:28,882 - INFO - Using playlist path format: artist_album
2026-05-08 09:58:28,882 - INFO - Format description: Artist(s)/Album/CD# - Track# - Artist(s) - Title - Album.ext
2026-05-08 09:58:28,882 - INFO - Step 1: Building library cache
2026-05-08 09:58:28,882 - INFO - Scanning music library: test_music_library
2026-05-08 09:58:28,882 - INFO - Processing album artist: 2Pac, Snoop Dogg
2026-05-08 09:58:28,900 - INFO - Processing album artist: A Tribe Called Quest
2026-05-08 09:58:28,901 - INFO - Processing album artist: Ciara, Missy Elliott
2026-05-08 09:58:28,901 - INFO - Processing album artist: Die Ärzte
2026-05-08 09:58:28,902 - INFO - Processing album artist: Gorillaz
2026-05-08 09:58:28,902 - INFO - Processing album artist: Jamiroquai
2026-05-08 09:58:28,910 - INFO - Processing album artist: JAY-Z, Beyoncé
2026-05-08 09:58:28,911 - INFO - Processing album artist: Santana
2026-05-08 09:58:29,003 - INFO - Processing album artist: The Offspring
2026-05-08 09:58:29,029 

## Verify Results

Let's check the output playlist and unmatched log.

In [8]:
print("=" * 80)
print("OUTPUT PLAYLIST (test_output.m3u8)")
print("=" * 80)
if os.path.exists('test_output.m3u8'):
    with open('test_output.m3u8', 'r', encoding='utf-8') as f:
        content = f.read()
        print(content)
        print(f"\nTotal matched songs: {content.count('#EXTINF')}")
else:
    print("File not found!")

print("\n" + "=" * 80)
print("UNMATCHED LOG (test_unmatched.log)")
print("=" * 80)
if os.path.exists('test_unmatched.log'):
    with open('test_unmatched.log', 'r', encoding='utf-8') as f:
        print(f.read())
else:
    print("File not found!")

OUTPUT PLAYLIST (test_output.m3u8)
#EXTM3U
#EXTINF:135,The Offspring - (Can't Get My) Head Around You
The Offspring\Splinter\1 - 164 - (Can't Get My) Head Around You - The Offspring - Splinter.flac
#EXTINF:352,Santana - (Da Le) Yaleo
Santana\Supernatural (Remastered)\1 - 208 - (Da Le) Yaleo - Santana - Supernatural (Remastered).flac
#EXTINF:300,Jamiroquai - (Don't) Give Hate a Chance
Jamiroquai\Dynamite\1 - 247 - (Don't) Give Hate a Chance - Jamiroquai - Dynamite.flac
#EXTINF:223,The Rolling Stones - (I Can't Get No) Satisfaction - Mono
The Rolling Stones\Out Of Our Heads\1 - 458 - (I Can't Get No) Satisfaction - Mono - The Rolling Stones - Out Of Our Heads.flac
#EXTINF:206,JAY-Z, Beyoncé - 03' Bonnie & Clyde
JAY-Z, Beyoncé\The Blueprint 2 The Gift & The Curse\1 - 1 - 03' Bonnie & Clyde - JAY-Z, Beyoncé - The Blueprint 2 The Gift & The Curse.flac
#EXTINF:235,Die Ärzte - 1/2 Lovesong
Die Ärzte\13\1 - 2 - 1∕2 Lovesong - Die Ärzte - 13.flac
#EXTINF:204,Ciara, Missy Elliott - 1, 2 Step (fe

## Verify File Structure

Let's verify the created library structure.

In [9]:
print("Test Music Library Structure:\n")
for root, dirs, files in os.walk('test_music_library'):
    level = root.replace('test_music_library', '').count(os.sep)
    indent = ' ' * 2 * level
    print(f"{indent}{os.path.basename(root)}/")
    subindent = ' ' * 2 * (level + 1)
    for file in sorted(files):
        print(f"{subindent}{file}")

Test Music Library Structure:

test_music_library/
  2Pac, Snoop Dogg/
    All Eyez On Me/
      1 - 8 - 2 Of Amerikaz Most Wanted (ft. Snoop Doggy Dogg) - 2Pac, Snoop Dogg - All Eyez On Me.flac
  A Tribe Called Quest/
    From NYC/
      1 - 5 - 1nce again - A Tribe Called Quest - From NYC.flac
  Ciara, Missy Elliott/
    Goodies/
      1 - 3 - 1, 2 Step (feat. Missy Elliott) - Ciara, Missy Elliott - Goodies.flac
  Die Ärzte/
    13/
      1 - 2 - 1∕2 Lovesong - Die Ärzte - 13.flac
  Gorillaz/
    Gorillaz/
      1 - 4 - 19-2000 - Soulchild Remix - Gorillaz - Gorillaz.flac
  Jamiroquai/
    Dynamite/
      1 - 247 - (Don't) Give Hate a Chance - Jamiroquai - Dynamite.flac
  JAY-Z, Beyoncé/
    The Blueprint 2 The Gift & The Curse/
      1 - 1 - 03' Bonnie & Clyde - JAY-Z, Beyoncé - The Blueprint 2 The Gift & The Curse.flac
  Santana/
    Supernatural (Remastered)/
      1 - 208 - (Da Le) Yaleo - Santana - Supernatural (Remastered).flac
  The Offspring/
    Splinter/
      1 - 164 - (Ca

## Test Simple Text Playlist Format

Now let's test the script with a simple text playlist format (Artist - Title per line).

In [10]:
def create_text_playlist(entries, output_path='test_text_playlist.txt'):
    """Create a simple text playlist with Artist - Title format"""
    with open(output_path, 'w', encoding='utf-8') as f:
        for entry in entries:
            f.write(f"{entry['artist']} - {entry['title']}\n")
    
    print(f"Created text playlist: {output_path}")
    print(f"\nContents:")
    with open(output_path, 'r', encoding='utf-8') as f:
        content = f.read()
        print(content)
    return output_path

create_text_playlist(entries)

Created text playlist: test_text_playlist.txt

Contents:
The Offspring - (Can't Get My) Head Around You
Santana - (Da Le) Yaleo
Jamiroquai - (Don't) Give Hate a Chance
The Rolling Stones - (I Can't Get No) Satisfaction - Mono
JAY-Z, Beyoncé - 03' Bonnie & Clyde
Die Ärzte - 1/2 Lovesong
Ciara, Missy Elliott - 1, 2 Step (feat. Missy Elliott)
Gorillaz - 19-2000 - Soulchild Remix
A Tribe Called Quest - 1nce again
2Pac, Snoop Dogg - 2 Of Amerikaz Most Wanted (ft. Snoop Doggy Dogg)



'test_text_playlist.txt'

## Run Playlist Matcher on Text Format

Test the script with the simple text playlist.

In [11]:
# Run the playlist matcher on text format
!python3 playlist_matcher.py \
    --playlist test_text_playlist.txt \
    --music-dir test_music_library \
    --output test_text_output.m3u8 \
    --log test_text_unmatched.log \
    --format artist_album

Python was not found; run without arguments to install from the Microsoft Store, or disable this shortcut from Settings > Apps > Advanced app settings > App execution aliases.


## Verify Text Playlist Results

Check the output from the text playlist conversion.

In [12]:
print("=" * 80)
print("TEXT PLAYLIST OUTPUT (test_text_output.m3u8)")
print("=" * 80)
if os.path.exists('test_text_output.m3u8'):
    with open('test_text_output.m3u8', 'r', encoding='utf-8') as f:
        content = f.read()
        print(content)
        print(f"\nTotal matched songs: {content.count('#EXTINF')}")
else:
    print("File not found!")

print("\n" + "=" * 80)
print("TEXT PLAYLIST UNMATCHED LOG (test_text_unmatched.log)")
print("=" * 80)
if os.path.exists('test_text_unmatched.log'):
    with open('test_text_unmatched.log', 'r', encoding='utf-8') as f:
        log_content = f.read()
        print(log_content)
        if 'Unmatched: 0' in log_content:
            print("\n✓ SUCCESS: All songs from text playlist were matched!")
else:
    print("File not found!")

TEXT PLAYLIST OUTPUT (test_text_output.m3u8)
File not found!

TEXT PLAYLIST UNMATCHED LOG (test_text_unmatched.log)
File not found!


## Summary

This notebook demonstrates:
1. Creating a mock music library with proper metadata
2. Testing M3U8 playlist format matching
3. Testing simple text playlist format (Artist - Title) matching
4. Verifying that both formats produce correct M3U8 output playlists

Both tests should show 100% match rate (10/10 songs matched).

In [13]:
reload(pm)
base = pm.PlaylistMatcher(
    playlist_path=Path(""),
    music_dir="E:\\Music",
    output_path=Path(""),
    log_path=Path("unmatched.log"),
    path_format="artist_album"
)

2026-05-08 09:58:29,592 - INFO - Using playlist path format: artist_album
2026-05-08 09:58:29,592 - INFO - Format description: Artist(s)/Album/CD# - Track# - Artist(s) - Title - Album.ext


In [14]:
base.build_library_cache()

2026-05-08 09:58:29,597 - INFO - Step 1: Building library cache
2026-05-08 09:58:29,597 - INFO - Scanning music library: E:\Music
2026-05-08 09:58:29,915 - INFO - Processing album artist: 19 Naughty III (US Release)
2026-05-08 09:58:30,108 - INFO - Processing album artist: 2 Brothers On The 4th Floor
2026-05-08 09:58:30,330 - INFO - Processing album artist: 2 Unlimited
2026-05-08 09:58:36,426 - INFO - Processing album artist: 2Pac
2026-05-08 09:58:39,673 - INFO - Cached 100 files...
2026-05-08 09:58:41,449 - INFO - Processing album artist: 3 Steps Ahead
2026-05-08 09:58:41,533 - INFO - Processing album artist: 4am Kru
2026-05-08 09:58:41,674 - INFO - Processing album artist: 50 Cent
2026-05-08 09:58:45,242 - INFO - Cached 200 files...
2026-05-08 09:58:47,265 - INFO - Processing album artist: 6 Pack (Explicit Version)
2026-05-08 09:58:47,313 - INFO - Processing album artist: 702
2026-05-08 09:58:47,360 - INFO - Processing album artist: 808 State
2026-05-08 09:58:47,435 - INFO - Processi

In [ ]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Favourites.m3u8")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Favourites.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Favourites.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 10:26:18,460 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Favourites.m3u8
2026-05-08 10:26:18,462 - INFO - Read 2129 lines from playlist
2026-05-08 10:26:18,462 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 10:26:18,463 - INFO - Detected playlist format: m3u8
2026-05-08 10:26:19,429 - INFO - Attempting partial match for artist 'Miami Sound Machine'
2026-05-08 10:26:19,429 - INFO -   Searching all tracks by artist 'Miami Sound Machine'
2026-05-08 10:26:19,430 - WARNING - ✗ No match: Miami Sound Machine - Conga!
2026-05-08 10:26:19,430 - WARNING -   Reason: No files found with title 'Conga!'; Found 1 file(s) with matching artist; No files found with album 'The Very Best Of Gloria Estefan (English Version)'; Artist exists but with different title
2026-05-08 10:26:20,252 - INFO - Attempting partial match for artist 'The Notorious B.I.G.'
2026-05-08 10:26:20,253 - INFO -   Searching all tracks by artist 'The Notorious B.I.G.'
2026-05-08 10:26:20,254 

In [45]:
base.output_path = Path("E:\\Playlists\\copy\\Playlists\\Favourites.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 11:26:24,685 - INFO - Step 4: Writing new playlist: E:\Playlists\copy\Playlists\Favourites.m3u8
2026-05-08 11:26:24,688 - INFO - Wrote 1063 entries to new playlist


In [ ]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\How Hard Can It Be.m3u8")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\How Hard Can It Be.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\How Hard Can It Be.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 11:56:35,281 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\How Hard Can It Be.m3u8
2026-05-08 11:56:35,391 - INFO - Read 187 lines from playlist
2026-05-08 11:56:35,391 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 11:56:35,392 - INFO - Detected playlist format: m3u8
2026-05-08 11:56:35,824 - INFO - Matched: 93, Unmatched: 0
2026-05-08 11:56:35,825 - INFO - Step 4: Writing new playlist: E:\rockbox\library-playlist-parser\How Hard Can It Be.m3u8
2026-05-08 11:56:35,826 - INFO - Wrote 93 entries to new playlist
2026-05-08 11:56:35,826 - INFO - Step 5: Writing unmatched log: E:\rockbox\library-playlist-parser\unmatched.log
2026-05-08 11:56:35,827 - INFO - 
2026-05-08 11:56:35,827 - INFO - SUMMARY
2026-05-08 11:56:35,827 - INFO - ================================================================================
2026-05-08 11:56:35,828 - INFO - Total songs: 93
2026-05-08 11:56:35,828 - INFO - Matched: 93
2026-05-08 11:56:35,828 - INFO - Unmatched: 0
202

(0/92)  File already exists, skipping: E:\Playlists\copy\Music\Phil Collins\Love Songs (A Compilation Old and New)\1 - 4 - A Groovy Kind of Love - Phil Collins - Love Songs (A Compilation Old and New).flac
(1/92)  Copying: E:\Music\The 1975\Being Funny In A Foreign Language\1 - 74 - About You - The 1975 - Being Funny In A Foreign Language.flac -> E:\Playlists\copy\Music\The 1975\Being Funny In A Foreign Language\1 - 74 - About You - The 1975 - Being Funny In A Foreign Language.flac
(2/92)  Copying: E:\Music\Harry Styles\Fine Line\1 - 3 - Adore You - Harry Styles - Fine Line.flac -> E:\Playlists\copy\Music\Harry Styles\Fine Line\1 - 3 - Adore You - Harry Styles - Fine Line.flac
(3/92)  Copying: E:\Music\Taylor Swift\All Of The Girls You Loved Before\1 - 25 - All Of The Girls You Loved Before - Taylor Swift - All Of The Girls You Loved Before.flac -> E:\Playlists\copy\Music\Taylor Swift\All Of The Girls You Loved Before\1 - 25 - All Of The Girls You Loved Before - Taylor Swift - All Of T

2026-05-08 11:57:09,210 - INFO - Step 4: Writing new playlist: E:\Playlists\copy\Playlists\How Hard Can It Be.m3u8
2026-05-08 11:57:09,212 - INFO - Wrote 93 entries to new playlist


In [ ]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Wedding Party.m3u8")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Wedding Party.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Wedding Party.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 11:38:29,764 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Wedding Party.m3u8
2026-05-08 11:38:29,765 - INFO - Read 555 lines from playlist
2026-05-08 11:38:29,766 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 11:38:29,767 - INFO - Detected playlist format: m3u8
2026-05-08 11:38:31,266 - INFO - Matched: 277, Unmatched: 0
2026-05-08 11:38:31,267 - INFO - Step 4: Writing new playlist: E:\rockbox\library-playlist-parser\Wedding Party.m3u8
2026-05-08 11:38:31,269 - INFO - Wrote 277 entries to new playlist
2026-05-08 11:38:31,269 - INFO - Step 5: Writing unmatched log: E:\rockbox\library-playlist-parser\unmatched.log
2026-05-08 11:38:31,270 - INFO - 
2026-05-08 11:38:31,270 - INFO - SUMMARY
2026-05-08 11:38:31,270 - INFO - ================================================================================
2026-05-08 11:38:31,271 - INFO - Total songs: 277
2026-05-08 11:38:31,271 - INFO - Matched: 277
2026-05-08 11:38:31,271 - INFO - Unmatched: 0
2026-05-0

(0/276)  Copying: E:\Music\The Rolling Stones\Hot Rocks 1964-1971\1 - 35 - (I Can't Get No) Satisfaction - Mono - The Rolling Stones - Hot Rocks 1964-1971.flac -> E:\Playlists\copy\Playlists\The Rolling Stones\Hot Rocks 1964-1971\1 - 35 - (I Can't Get No) Satisfaction - Mono - The Rolling Stones - Hot Rocks 1964-1971.flac
(1/276)  Copying: E:\Music\Various Artists\Dirty Dancing (Original Motion Picture Soundtrack)\1 - 232 - (I've Had) The Time Of My Life - From Dirty Dancing Soundtrack - Bill Medley, Jennifer Warnes - Dirty Dancing (Original Motion Picture Soundtrack).flac -> E:\Playlists\copy\Playlists\Various Artists\Dirty Dancing (Original Motion Picture Soundtrack)\1 - 232 - (I've Had) The Time Of My Life - From Dirty Dancing Soundtrack - Bill Medley, Jennifer Warnes - Dirty Dancing (Original Motion Picture Soundtrack).flac
(2/276)  Copying: E:\Music\Vanessa Carlton\Best Of\1 - 1 - A Thousand Miles - Vanessa Carlton - Best Of.flac -> E:\Playlists\copy\Playlists\Vanessa Carlton\Best

2026-05-08 11:41:52,095 - INFO - Step 4: Writing new playlist: E:\Playlists\copy\Playlists\Wedding Party.m3u8
2026-05-08 11:41:52,097 - INFO - Wrote 277 entries to new playlist


In [ ]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Japanese Sunset.m3u8")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Japanese Sunset.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Japanese Sunset.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 11:50:29,426 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Japanese Sunset.m3u8
2026-05-08 11:50:29,481 - INFO - Read 517 lines from playlist
2026-05-08 11:50:29,482 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 11:50:29,482 - INFO - Detected playlist format: m3u8
2026-05-08 11:50:31,524 - INFO - Matched: 258, Unmatched: 0
2026-05-08 11:50:31,525 - INFO - Step 4: Writing new playlist: E:\rockbox\library-playlist-parser\Japanese Sunset.m3u8
2026-05-08 11:50:31,794 - INFO - Wrote 258 entries to new playlist
2026-05-08 11:50:31,795 - INFO - Step 5: Writing unmatched log: E:\rockbox\library-playlist-parser\unmatched.log
2026-05-08 11:50:31,795 - INFO - 
2026-05-08 11:50:31,796 - INFO - SUMMARY
2026-05-08 11:50:31,796 - INFO - ================================================================================
2026-05-08 11:50:31,796 - INFO - Total songs: 258
2026-05-08 11:50:31,797 - INFO - Matched: 258
2026-05-08 11:50:31,797 - INFO - Unmatched: 0
2026-

(0/257)  Copying: E:\Music\haruka nakamura\grace\1 - 1 - every day - haruka nakamura, Janis Crunch - grace.flac -> E:\Playlists\copy\Playlists\haruka nakamura\grace\1 - 1 - every day - haruka nakamura, Janis Crunch - grace.flac
(1/257)  Copying: E:\Music\haruka nakamura\grace\1 - 2 - arne - haruka nakamura - grace.flac -> E:\Playlists\copy\Playlists\haruka nakamura\grace\1 - 2 - arne - haruka nakamura - grace.flac
(2/257)  Copying: E:\Music\haruka nakamura\grace\1 - 3 - opus - haruka nakamura - grace.flac -> E:\Playlists\copy\Playlists\haruka nakamura\grace\1 - 3 - opus - haruka nakamura - grace.flac
(3/257)  Copying: E:\Music\haruka nakamura\grace\1 - 4 - ralgo - haruka nakamura, Janis Crunch - grace.flac -> E:\Playlists\copy\Playlists\haruka nakamura\grace\1 - 4 - ralgo - haruka nakamura, Janis Crunch - grace.flac
(4/257)  Copying: E:\Music\haruka nakamura\grace\1 - 5 - elm - haruka nakamura - grace.flac -> E:\Playlists\copy\Playlists\haruka nakamura\grace\1 - 5 - elm - haruka nakamu

2026-05-08 11:53:09,797 - INFO - Step 4: Writing new playlist: E:\Playlists\copy\Playlists\Japanese Sunset.m3u8
2026-05-08 11:53:09,799 - INFO - Wrote 258 entries to new playlist


In [19]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\So.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\So.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\So.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 10:20:02,615 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\So.txt
2026-05-08 10:20:02,649 - INFO - Read 225 lines from playlist
2026-05-08 10:20:02,650 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 10:20:02,650 - INFO - Detected playlist format: text
2026-05-08 10:20:02,819 - WARNING - ✗ No match: Silk City,Dua Lipa,Mark Ronson,Diplo - Electricity
2026-05-08 10:20:02,820 - WARNING -   Reason: Found 2 file(s) with matching title; No files found with artist 'Silk City,Dua Lipa,Mark Ronson,Diplo'; Title exists but with different artist
2026-05-08 10:20:03,230 - WARNING - ✗ No match: Childish Gambino,Brittany Howard - Stay High - Childish Gambino Version
2026-05-08 10:20:03,231 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Childish Gambino,Brittany Howard'; Title exists but with different artist
2026-05-08 10:20:03,418 - INFO - Attempting partial match for artist 'Joesef'
2026-05-08 10:20:03,419 - INFO -   Sea

In [20]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Wee rock.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Wee rock.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Wee rock.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 10:20:04,918 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Wee rock.txt
2026-05-08 10:20:04,952 - INFO - Read 48 lines from playlist
2026-05-08 10:20:04,953 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 10:20:04,953 - INFO - Detected playlist format: text
2026-05-08 10:20:05,302 - INFO - Matched: 48, Unmatched: 0
2026-05-08 10:20:05,303 - INFO - Step 4: Writing new playlist: E:\rockbox\library-playlist-parser\Wee rock.m3u8
2026-05-08 10:20:05,312 - INFO - Wrote 48 entries to new playlist
2026-05-08 10:20:05,312 - INFO - Step 5: Writing unmatched log: E:\rockbox\library-playlist-parser\unmatched.log
2026-05-08 10:20:05,336 - INFO - 
2026-05-08 10:20:05,337 - INFO - SUMMARY
2026-05-08 10:20:05,338 - INFO - ================================================================================
2026-05-08 10:20:05,338 - INFO - Total songs: 48
2026-05-08 10:20:05,338 - INFO - Matched: 48
2026-05-08 10:20:05,338 - INFO - Unmatched: 0
2026-05-08 10:20:05,339 -

In [21]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Marvelous.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Marvelous.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Marvelous.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 10:20:05,345 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Marvelous.txt
2026-05-08 10:20:05,475 - INFO - Read 217 lines from playlist
2026-05-08 10:20:05,475 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 10:20:05,476 - INFO - Detected playlist format: text
2026-05-08 10:20:05,493 - WARNING - ✗ No match: Hazey Eyes,Panama - Emotion
2026-05-08 10:20:05,493 - WARNING -   Reason: Found 2 file(s) with matching title; No files found with artist 'Hazey Eyes,Panama'; Title exists but with different artist
2026-05-08 10:20:05,572 - WARNING - ✗ No match: Darlingside - Hold Your Head Up High - Recorded at Spotify Studios NYC
2026-05-08 10:20:05,573 - WARNING -   Reason: No files found with title 'Hold Your Head Up High - Recorded at Spotify Studios NYC'; No files found with artist 'Darlingside'
2026-05-08 10:20:05,678 - WARNING - ✗ No match: Sara Bareilles,Ingrid Michaelson - Winter Song
2026-05-08 10:20:05,678 - WARNING -   Reason: Found 2 file(s) with ma

In [22]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Christmas .txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Christmas.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Christmas.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 10:20:07,237 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Christmas .txt
2026-05-08 10:20:07,259 - INFO - Read 36 lines from playlist
2026-05-08 10:20:07,259 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 10:20:07,259 - INFO - Detected playlist format: text
2026-05-08 10:20:07,339 - WARNING - ✗ No match: Ingrid Michaelson,Grace VanderWaal - Rockin' Around The Christmas Tree
2026-05-08 10:20:07,340 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Ingrid Michaelson,Grace VanderWaal'; Title exists but with different artist
2026-05-08 10:20:07,349 - WARNING - ✗ No match: Miley Cyrus,Mark Ronson,Sean Ono Lennon - Happy Xmas (War Is Over) (feat. Sean Ono Lennon)
2026-05-08 10:20:07,350 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Miley Cyrus,Mark Ronson,Sean Ono Lennon'; Title exists but with different artist
2026-05-08 10:20:07,425 - WARNING - ✗ No match: Bing Crosby,The A

In [23]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Nachtmusik.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Nachtmusik.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Nachtmusik.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 10:20:07,569 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Nachtmusik.txt
2026-05-08 10:20:07,585 - INFO - Read 117 lines from playlist
2026-05-08 10:20:07,586 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 10:20:07,586 - INFO - Detected playlist format: text
2026-05-08 10:20:07,896 - WARNING - ✗ No match: Broken Bells,Danger Mouse,James Mercer - Holding on for Life
2026-05-08 10:20:07,897 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Broken Bells,Danger Mouse,James Mercer'; Title exists but with different artist
2026-05-08 10:20:07,908 - WARNING - ✗ No match: Broken Bells,Danger Mouse,James Mercer - The High Road
2026-05-08 10:20:07,909 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Broken Bells,Danger Mouse,James Mercer'; Title exists but with different artist
2026-05-08 10:20:07,920 - WARNING - ✗ No match: Broken Bells,Danger Mouse,James Mercer - October
2026-05-08

In [24]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\Sing Along .txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\Sing Along.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

lines = base.read_old_playlist()
matched, unmatched = base.find_matches(lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\Sing Along.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 10:20:08,587 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\Sing Along .txt
2026-05-08 10:20:08,606 - INFO - Read 102 lines from playlist
2026-05-08 10:20:08,607 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 10:20:08,608 - INFO - Detected playlist format: text
2026-05-08 10:20:08,668 - WARNING - ✗ No match: Whitney Houston,George Michael - If I Told You That (feat. George Michael) - Radio Edit
2026-05-08 10:20:08,669 - WARNING -   Reason: No files found with title 'If I Told You That (feat. George Michael) - Radio Edit'; No files found with artist 'Whitney Houston,George Michael'
2026-05-08 10:20:08,806 - WARNING - ✗ No match: Troy,Gabriella,Disney - Start of Something New
2026-05-08 10:20:08,806 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Troy,Gabriella,Disney'; Title exists but with different artist
2026-05-08 10:20:08,819 - WARNING - ✗ No match: Troy,Gabriella,Disney - Breaking Free
2026-05-08 10:20:08

In [25]:
base.playlist_path = Path("E:\\rockbox\\Playlists\\SAMURAI CHAMPLOO.txt")
base.output_path = Path("E:\\rockbox\\library-playlist-parser\\SAMURAI CHAMPLOO.m3u8")
base.log_path = Path("E:\\rockbox\\library-playlist-parser\\unmatched.log")
base.path_prefix = ''

hhcib_lines = base.read_old_playlist()
matched, unmatched = base.find_matches(hhcib_lines)
base.write_new_playlist(matched)
base.write_log(matched, unmatched)

base.output_path = Path("E:\\Playlists\\SAMURAI CHAMPLOO.m3u8")
base.path_prefix = '..\\Music\\'
base.write_new_playlist(matched)
base.path_prefix = ''

2026-05-08 10:20:09,499 - INFO - Step 2: Reading playlist: E:\rockbox\Playlists\SAMURAI CHAMPLOO.txt
2026-05-08 10:20:09,509 - INFO - Read 78 lines from playlist
2026-05-08 10:20:09,510 - INFO - Step 3: Finding matches for playlist entries
2026-05-08 10:20:09,511 - INFO - Detected playlist format: text
2026-05-08 10:20:09,527 - WARNING - ✗ No match: Nujabes,Shing02 - battlecry
2026-05-08 10:20:09,528 - WARNING -   Reason: Found 1 file(s) with matching title; No files found with artist 'Nujabes,Shing02'; Title exists but with different artist
2026-05-08 10:20:09,543 - INFO - Attempting partial match for artist 'MINMI'
2026-05-08 10:20:09,543 - INFO -   Searching all tracks by artist 'MINMI'
2026-05-08 10:20:09,544 - WARNING - ✗ No match: MINMI - 四季ノ唄
2026-05-08 10:20:09,544 - WARNING -   Reason: No files found with title '四季ノ唄'; Found 1 file(s) with matching artist; Artist exists but with different title
2026-05-08 10:20:09,564 - WARNING - ✗ No match: TSUTCHIE,kazami - YOU feat. kazami
